# Create MERlin data organization

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent          # MERci/ (notebook lives in MERci/notebooks/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT027_saving_time/
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR = SAMPLE_DIR / "settings"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.data_organization import create_data_organization
from MERci.acquisition.dave import annotate_dave_with_round_info

MICROSCOPE  = "MF3"
SAMPLE_NAME = SAMPLE_DIR.name

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")

## Auto-detect frame tables

Lists all `frame_table_*.csv` files in the metadata folder. The bits frame table
is auto-selected as the most recently modified one that does **not** contain 405 nm.
Override `BITS_FT` / `CELLS_FT` manually if needed.

In [ ]:
ft_files = sorted(
    METADATA_DIR.glob("frame_table_*.csv"),
    key=lambda p: p.stat().st_mtime, reverse=True
)

print("Frame tables found (newest first):")
for f in ft_files:
    ft = pd.read_csv(f, index_col=0)
    colors = sorted(ft["color"].dropna().unique().astype(int))
    print(f"  {f.name}  →  colors: {colors}")

# Auto-pick: bits = most recent without 405; cells = most recent with 405
BITS_FT  = next(f for f in ft_files
                if 405 not in pd.read_csv(f, index_col=0)["color"].dropna().unique())
CELLS_FT = next(f for f in ft_files
                if 405 in pd.read_csv(f, index_col=0)["color"].dropna().unique())

print(f"\nBits  frame table : {BITS_FT.name}")
print(f"Cells frame table : {CELLS_FT.name}")

## Round – bit – color mapping

Define which bit is imaged in each round and at which wavelength.
Edit this table to match the actual experiment codebook.

Columns saved to `round_bit_color_map.csv`:
- `round`  : imaging round (1-indexed)
- `bit`    : bit number
- `color`  : excitation wavelength in nm

In [ ]:
# (round, bit_number, color_nm)
round_bit_color = [
    (1,  1,  750), (1,  2,  650), (1,  17, 560),
    (2,  3,  750), (2,  4,  650), (2,  18, 560),
    (3,  5,  750), (3,  6,  650), (3,  19, 560),
    (4,  7,  750), (4,  8,  650), (4,  20, 560),
    (5,  9,  750), (5,  10, 650), (5,  21, 560),
    (6,  11, 750), (6,  12, 650), (6,  22, 560),
    (7,  13, 750), (7,  14, 650), (7,  23, 560),
    (8,  15, 750), (8,  16, 650), (8,  24, 560),
    (9,  25, 750), (9,  26, 650), (9,  27, 560),
]

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)
print(f"Saved: {rbc_path}")
print(rbc_df.to_string(index=False))

## Series patterns from round_info.csv

Reads the bits and cells series patterns so the correct `imageType` and
`imageRegExp` fields are written into the data organization.  
Re-run notebook 03 first if `round_info.csv` is out of date.

In [ ]:
round_info_path = METADATA_DIR / "round_info.csv"
round_info      = pd.read_csv(round_info_path)

bits_series  = round_info[~round_info["series"].str.contains("cells")].iloc[0]["series"]
cells_series = round_info[ round_info["series"].str.contains("cells")].iloc[0]["series"]

print(f"Bits  series : {bits_series}")
print(f"Cells series : {cells_series}")

## Build and save data organization

In [ ]:
readouts  = pd.read_csv(SAMPLE_DIR / "readouts.csv")
ft_bits   = pd.read_csv(BITS_FT,  index_col=0)
ft_cells  = pd.read_csv(CELLS_FT, index_col=0)

data_org = create_data_organization(
    bits_frame_table  = ft_bits,
    cells_frame_table = ft_cells,
    round_bit_color   = round_bit_color,
    readouts          = readouts,
    bits_series       = bits_series,
    cells_series      = cells_series,
    include_dapi      = True,
)

out_name = f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
out_path = METADATA_DIR / out_name
data_org.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"\n{len(data_org)} rows  ({len(data_org)-1} bits + DAPI)")
data_org

## Annotate Dave XML with bit information

Adds per-round XML comments to the Dave config generated by notebook 03.
- Round 1: comment before the imaging loop
- Rounds 2+: comment before the fluidics loop that precedes that imaging round

Re-run this cell whenever the round–bit–color mapping changes.

In [ ]:
dave_files = sorted(SETTINGS_DIR.glob("dave-*.xml"))
if not dave_files:
    print("No dave-*.xml found in settings/ — run notebook 03 first.")
else:
    dave_path = dave_files[-1]   # most recently generated
    print(f"Annotating: {dave_path.name}")
    annotate_dave_with_round_info(dave_path, round_bit_color)
    print("Done. Preview of annotated file:")
    with open(dave_path, encoding="ISO-8859-1") as fh:
        print(fh.read())